In [6]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

from metrics import (
    F1_score,
    IoU,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
gt_videos_path = Path(
    "../benchmark/queries_and_videos_test.json"
)

all_possible_videos = list([f.stem for f in Path(
    "/media/EVO870/datasets/prompting-mammalps-v2/annotations_2023/test"
).rglob("*.json")])

retrieved_results_path = Path(
    "/media/eceo_scratch_haas001/results/prompting_mammalps-v2/LLM/qwen/Qwen3-8B-oracle_2023_orig.json"
)  # Path to the JSON file create by run.py

with open(gt_videos_path, "r") as f:
    gt_videos_cat = json.load(f)

gt_videos = {}
for cat in gt_videos_cat:
    
    for query, video_list  in gt_videos_cat[cat].items():
        gt_videos[query] = video_list

with open(retrieved_results_path, "r") as f:
    pred_videos = json.load(f)


with open("../benchmark/query_categories.json", "r") as f:
    query_categories = json.load(f)

query2eco_cat = {q: [] for q in gt_videos.keys()}
for cat in query_categories["ECOLOGY"]:
    for q in query_categories["ECOLOGY"][cat]:
        query2eco_cat[q].append(cat)

query2cv_cat = {q: [] for q in gt_videos.keys()}
for cat in query_categories["VISION"]:
    for q in query_categories["VISION"][cat]:
        query2cv_cat[q].append(cat)

# Overall performance

In [12]:
mF1, F1_queries = F1_score(gt_queries_videos=gt_videos, pred_queries_videos=pred_videos, all_videos=all_possible_videos)
mIoU, IoU_queries = IoU(gt_queries_videos=gt_videos, pred_queries_videos=pred_videos, all_videos=all_possible_videos)

In [13]:
print(f"F1-score (macro-avg.): {mF1:.2f}")
print(f"mean IoU: {mIoU:.2f}")

F1-score (macro-avg.): 0.29
mean IoU: 0.24


In [14]:
print(f"F1-score (macro-avg.) w/o Ref prompts: {np.nanmean([s for q, s in F1_queries.items() if '<vid>' not in q]):.2f}")
print(f"mean IoU w/o Ref prompts : {np.nanmean([s for q, s in IoU_queries.items() if '<vid>' not in q]):.2f}")

F1-score (macro-avg.) w/o Ref prompts: 0.32
mean IoU w/o Ref prompts : 0.27


# Performance per query et per category

In [13]:
results_df = pd.concat([
    pd.DataFrame.from_dict(IoU_queries, orient="index", columns=["mIoU"]),
    pd.DataFrame.from_dict(F1_queries, orient="index", columns=["F1-score"]),
    ], axis=1)

pd.set_option('display.max_rows', 150)
pd.set_option('display.max_colwidth', 90)
results_df.sort_values("mIoU", ascending=False)

,mIoU,F1-score
A red deer participating in courtship in rainy weather.,1.000000,1.000000
A roe deer participating in courtship in rainy weather.,1.000000,1.000000
A red deer resting in rainy weather.,1.000000,1.000000
A juvenile roe deer.,1.000000,1.000000
A fox chasing prey.,1.000000,1.000000
A fox sniffing.,1.000000,1.000000
A juvenile roe deer preparing to suckle.,1.000000,1.000000
An individual doing the same sequence of actions as the individual of <vid>S2_C1_E323_V0074</vid>.,1.000000,1.000000
A fox reacting to a camera.,1.000000,1.000000
A red deer vocalizing in rainy weather.,1.000000,1.000000


In [14]:
len(results_df[results_df[["F1-score"]]==0])

135

In [52]:
eco_cat_df = pd.DataFrame([query2eco_cat], index=["eco_cat"]).T
cv_cat_df = pd.DataFrame([query2cv_cat], index=["cv_cat"]).T
results_cat_df = results_df.merge(eco_cat_df, left_index=True, right_index=True).merge(cv_cat_df, left_index=True, right_index=True)

In [53]:
# Ecology categories
results_cat_df[["mIoU", "F1-score", "eco_cat"]].explode(["eco_cat"]).groupby("eco_cat").mean().T

eco_cat,CAMERA_REACTION,COMMON,COURTSHIP,RARE,SOCIAL
mIoU,0.147499,0.759409,0.248798,0.343454,0.286984
F1-score,0.190382,0.858781,0.311536,0.427785,0.330484


In [54]:
results_cat_df[["mIoU", "F1-score", "eco_cat"]].explode(["eco_cat"]).groupby("eco_cat").count().T

eco_cat,CAMERA_REACTION,COMMON,COURTSHIP,RARE,SOCIAL
mIoU,28,2,27,48,36
F1-score,28,2,27,48,36


In [55]:
results_cat_df[["mIoU", "F1-score", "cv_cat"]].explode(["cv_cat"]).groupby("cv_cat").mean().T

cv_cat,COMPLEX,MULTI_ATTR,MULTI_INDIV,SINGLE_ATTR,VIDEO_COMPARISON
mIoU,0.172718,0.306205,0.146442,0.264859,0.086783
F1-score,0.216052,0.371186,0.188554,0.353194,0.130371


In [56]:
results_cat_df[["mIoU", "F1-score", "cv_cat"]].explode(["cv_cat"]).groupby("cv_cat").count().T

cv_cat,COMPLEX,MULTI_ATTR,MULTI_INDIV,SINGLE_ATTR,VIDEO_COMPARISON
mIoU,15,92,25,14,22
F1-score,15,92,25,14,22


## Individual query

In [30]:
print("def check_file(file_id):\n    individual_tracks = get_tracks_from_file_id(file_id)\n    has_adult_red_deer = check_tracks_contain_deer_age(individual_tracks, age=DAge.ADULT, deer_species=Species.RED_DEER)\n    if not has_adult_red_deer:\n        return False\n    unique_activities = get_unique_activities_from_tracks(individual_tracks)\n    return unique_activities == {Activity.FORAGING}")

def check_file(file_id):
    individual_tracks = get_tracks_from_file_id(file_id)
    has_adult_red_deer = check_tracks_contain_deer_age(individual_tracks, age=DAge.ADULT, deer_species=Species.RED_DEER)
    if not has_adult_red_deer:
        return False
    unique_activities = get_unique_activities_from_tracks(individual_tracks)
    return unique_activities == {Activity.FORAGING}


In [32]:
q = "A single adult red deer foraging only."
print("Size of retrieved set", len(pred_videos[q]))
print("Size of GT set", len(gt_videos[q]))
print("True positives", len(set(pred_videos[q]).intersection(set(gt_videos[q]))))
print("Missing videos", len(set(gt_videos[q]) - set(pred_videos[q])))
print("false positives", len(set(pred_videos[q]) - set(gt_videos[q])))
print("True negatives", len((set(all_possible_videos) - set(gt_videos[q])).intersection(set(all_possible_videos) - set(pred_videos[q]))))

Size of retrieved set 356
Size of GT set 373
True positives 292
Missing videos 81
false positives 64
True negatives 338


In [33]:
print("Missing videos:")
set(gt_videos[q]) - set(pred_videos[q])

Missing videos:


{'S1_C1_E117_V0360',
 'S1_C2_E121_V0187',
 'S1_C2_E121_V0192',
 'S1_C2_E121_V0193',
 'S1_C2_E143_V0327',
 'S1_C2_E144_V0328',
 'S1_C2_E196_V0443',
 'S1_C4_F269_V0273',
 'S1_C4_F269_V0274',
 'S1_C4_F269_V0275',
 'S1_C4_F270_V0278',
 'S1_C4_F270_V0279',
 'S1_C4_F270_V0280',
 'S1_C4_F270_V0281',
 'S1_C4_F270_V0282',
 'S1_C4_F270_V0283',
 'S1_C4_F270_V0284',
 'S1_C4_F270_V0285',
 'S1_C4_F270_V0286',
 'S1_C4_F270_V0290',
 'S1_C4_F287_V0306',
 'S1_C4_F288_V0315',
 'S1_C4_F288_V0375',
 'S1_C4_F290_V0377',
 'S1_C6_F167_V0096',
 'S1_C6_F268_V0179',
 'S1_C6_F287_V0207',
 'S1_C6_F288_V0210',
 'S1_C6_F288_V0211',
 'S1_C6_F288_V0212',
 'S1_C6_F288_V0215',
 'S1_C6_F288_V0216',
 'S1_C6_F288_V0223',
 'S1_C6_F288_V0225',
 'S1_C6_F288_V0226',
 'S1_C6_F288_V0227',
 'S1_C6_F288_V0229',
 'S1_C6_F288_V0241',
 'S1_C6_F289_V0247',
 'S1_C6_F289_V0248',
 'S1_C6_F289_V0249',
 'S1_C6_F289_V0253',
 'S1_C6_F394_V0312',
 'S1_C6_F405_V0357',
 'S1_C6_F405_V0358',
 'S1_C6_F405_V0360',
 'S2_C1_E359_V0095',
 'S2_C1_E376_

In [34]:
print("False positives")

set(pred_videos[q]) - set(gt_videos[q])

False positives


{'S1_C1_E117_V0365',
 'S1_C1_E139_V0706',
 'S1_C1_E60_V0143',
 'S1_C1_E60_V0144',
 'S1_C2_E118_V0181',
 'S1_C3_E200_V0084',
 'S1_C4_F269_V0260',
 'S1_C4_F269_V0261',
 'S1_C4_F269_V0262',
 'S1_C4_F269_V0263',
 'S1_C4_F269_V0264',
 'S1_C4_F269_V0265',
 'S1_C4_F269_V0268',
 'S1_C4_F288_V0309',
 'S1_C4_F288_V0374',
 'S1_C5_F172_V0227',
 'S1_C5_F267_V0428',
 'S1_C6_F175_V0102',
 'S1_C6_F179_V0104',
 'S1_C6_F269_V0182',
 'S1_C6_F269_V0185',
 'S1_C6_F269_V0187',
 'S1_C6_F289_V0256',
 'S1_C6_F393_V0305',
 'S1_C6_F394_V0310',
 'S1_C6_F394_V0315',
 'S1_C6_F394_V0319',
 'S1_C6_F394_V0323',
 'S1_C6_F404_V0331',
 'S1_C6_F404_V0332',
 'S1_C6_F404_V0333',
 'S1_C6_F404_V0334',
 'S1_C6_F404_V0335',
 'S1_C6_F404_V0336',
 'S1_C6_F404_V0337',
 'S1_C6_F404_V0338',
 'S1_C6_F404_V0339',
 'S1_C6_F404_V0340',
 'S1_C6_F404_V0341',
 'S1_C6_F404_V0346',
 'S1_C6_F405_V0362',
 'S2_C1_E284_V0040',
 'S2_C1_E291_V0043',
 'S2_C1_E323_V0074',
 'S2_C1_E353_V0087',
 'S2_C1_E353_V0090',
 'S2_C1_E358_V0094',
 'S2_C1_F492_V0

In [35]:
print("Correctly retrieved")
set(pred_videos[q]).intersection(set(gt_videos[q]))

Correctly retrieved


{'S1_C1_E103_V0254',
 'S1_C1_E103_V0255',
 'S1_C1_E103_V0256',
 'S1_C1_E103_V0257',
 'S1_C1_E103_V0258',
 'S1_C1_E103_V0259',
 'S1_C1_E103_V0260',
 'S1_C1_E103_V0261',
 'S1_C1_E117_V0359',
 'S1_C1_E117_V0361',
 'S1_C1_E117_V0362',
 'S1_C1_E117_V0363',
 'S1_C1_E117_V0364',
 'S1_C1_E117_V0366',
 'S1_C1_E117_V0367',
 'S1_C1_E117_V0368',
 'S1_C1_E117_V0369',
 'S1_C1_E117_V0370',
 'S1_C1_E117_V0371',
 'S1_C1_E117_V0372',
 'S1_C1_E117_V0373',
 'S1_C1_E117_V0374',
 'S1_C1_E118_V0375',
 'S1_C1_E118_V0376',
 'S1_C1_E118_V0380',
 'S1_C1_E118_V0385',
 'S1_C1_E118_V0388',
 'S1_C1_E118_V0389',
 'S1_C1_E121_V0402',
 'S1_C1_E121_V0403',
 'S1_C1_E121_V0404',
 'S1_C1_E121_V0405',
 'S1_C1_E121_V0406',
 'S1_C1_E121_V0408',
 'S1_C1_E121_V0409',
 'S1_C1_E121_V0410',
 'S1_C1_E121_V0411',
 'S1_C1_E121_V0412',
 'S1_C1_E121_V0413',
 'S1_C1_E121_V0414',
 'S1_C1_E121_V0415',
 'S1_C1_E121_V0416',
 'S1_C1_E121_V0417',
 'S1_C1_E121_V0418',
 'S1_C1_E121_V0419',
 'S1_C1_E121_V0421',
 'S1_C1_E121_V0422',
 'S1_C1_E121_

# Joint table

In [27]:
salma_results = pd.read_csv("/media/eceo_scratch_haas001/results/prompting_mammalps-v2/LLM/qwen/Qwen3-8B-salma.csv").rename(columns={"Unnamed: 0":"query", "F1-score": "Agent+SALMA"}).set_index("query").drop("mIoU", axis=1)
oracle_results = pd.read_csv("/media/eceo_scratch_haas001/results/prompting_mammalps-v2/LLM/qwen/Qwen3-8B-oracle.csv").rename(columns={"Unnamed: 0":"query", "F1-score": "Agent+Oracle"}).set_index("query").drop("mIoU", axis=1)
clip_results = pd.read_csv("/media/eceo_scratch_haas001/results/prompting_mammalps-v2/CLIP4Clip/test/f1_queries.csv").rename(columns={"Unnamed: 0":"query", "F1-score": "CLIP4Clip"}).set_index("query")
intern_results = pd.read_csv("/media/eceo_scratch_haas001/results/prompting_mammalps-v2/InternVideo2/test/f1_queries.csv").rename(columns={"Unnamed: 0":"query", "F1-score": "InternVideo"}).set_index("query")

joint_results = (
    oracle_results
    .merge(salma_results, left_index=True, right_index=True)
    .merge(clip_results, left_index=True, right_index=True, how="left")
    .merge(intern_results, left_index=True, right_index=True, how="left")
    .sort_values(["Agent+Oracle", "Agent+SALMA", "CLIP4Clip", "InternVideo"], ascending=False)
)
joint_results

,Agent+Oracle,Agent+SALMA,CLIP4Clip,InternVideo
query,,,,
A fox reacting to a camera.,1.000000,1.000000,1.000000,1.000000
A juvenile roe deer suckling.,1.000000,1.000000,1.000000,0.997413
A juvenile roe deer preparing to suckle.,1.000000,1.000000,0.999354,0.996764
An adult female red deer escaping in sunny weather.,1.000000,1.000000,0.998061,0.998061
A fox sniffing.,1.000000,1.000000,0.984273,0.982940
A fox chasing prey.,1.000000,0.999354,1.000000,1.000000
A fox chasing prey in clear weather.,1.000000,0.999354,1.000000,1.000000
A roe deer participating in courtship in rainy weather.,1.000000,0.999354,0.027990,0.148148
A juvenile roe deer.,1.000000,0.998061,0.984273,0.965287


In [25]:
queries_w_vtag = [q.replace("<vid>", r"\vidtag{").replace("</vid>", r"}") for q in joint_results.index]

In [ ]:
print(*queries_w_vtag, sep="}\n"+r"\query{")

A juvenile roe deer preparing to suckle.}
\query{A fox sniffing.}
\query{A juvenile roe deer suckling.}
\query{An adult female red deer escaping in sunny weather.}
\query{A fox reacting to a camera.}
\query{A fox chasing prey.}
\query{A roe deer participating in courtship in rainy weather.}
\query{A fox chasing prey in clear weather.}
\query{A juvenile roe deer.}
\query{A wolf reacting to a camera.}
\query{An empty video.}
\query{A wolf chasing prey.}
\query{An animal resting.}
\query{A video of an animal lying down while resting.}
\query{A hare foraging.}
\query{A wolf chasing prey in a different weather condition from that in \vidtag{S2_C1_F573_V0093}.}
\query{An adult male red deer rubbing its antlers on the ground.}
\query{An animal being vigilant while the weather is rainy or overcast.}
\query{An animal bathing.}
\query{A hare.}
\query{An adult male red deer being vigilant after vocalizing.}
\query{An animal bathing while grooming.}
\query{Rainy weather.}
\query{An animal particip

In [29]:
print(*joint_results["InternVideo"].round(2).values, sep="\n")

1.0
1.0
1.0
1.0
0.98
1.0
1.0
0.15
0.97
0.57
0.15
0.1
0.4
0.0
0.01
nan
0.26
0.21
0.05
0.0
0.61
0.02
0.36
0.07
0.26
0.01
0.0
0.0
0.18
0.26
0.29
0.3
0.25
0.0
0.02
0.0
0.08
0.03
0.01
0.02
nan
0.25
0.13
nan
0.0
0.06
0.1
0.21
0.0
nan
0.02
0.0
0.08
0.09
0.33
0.09
nan
0.01
0.02
0.02
0.05
nan
nan
nan
0.0
0.0
0.0
0.01
0.01
0.01
0.0
0.01
0.0
0.11
0.02
0.02
0.01
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
0.99
0.14
0.33
0.6
nan
0.65
0.14
0.15
0.49
nan
0.04
0.08
0.07
0.34
0.0
0.17
0.0
0.09
0.16
0.0
nan
0.12
nan
0.21
0.05
0.0
0.03
0.04
0.03
nan
0.0
nan
0.0
nan
0.08
0.0
0.04
0.01
0.48
0.04
0.08
nan
nan
nan
